In [4]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from run_nn import run_single_baseline

HERE = Path.cwd()
PROJECT_DIR = HERE if (HERE / "preprocessing.py").exists() else Path("/Users/alibendaoud/Machine_Learning_CS/project1")
CACHE_DIR = HERE / "work" / "nn_cache"
RESULT_DIR = HERE / "nn_results"
print("Projet :", PROJECT_DIR)


Projet : /Users/alibendaoud/Machine_Learning_CS/project1


In [ ]:
# Paramètres à modifier
BASELINE_CONFIG = {
    "hidden_size": 64,        # neurones dans la couche cachée : essayer 32, 64, 128
    "learning_rate": 0.001,   # taux d’apprentissage : essayer 0.001 ou 0.0003
    "l2": 0.0001,              # régularisation : essayer 0.001 ou 0.0001
    "positive_weight": 3.0,   # poids des cas positifs : essayer 1, 2 ou 3
    "batch_size": 512,        # taille des mini-batches
    "epochs": 30,             # maximum ; l’arrêt anticipé choisit le meilleur
}

# À chaque lancement, le notebook choisit le prochain dossier libre
# (nn_results, nn_results_02, nn_results_03, ...), sans effacer les essais précédents.
RUN_BASELINE = True
if RUN_BASELINE:
    RESULT_DIR = HERE / "nn_results"
    if (RESULT_DIR / "selection.json").exists():
        index = 2
        while (HERE / f"nn_results_{index:02d}" / "selection.json").exists():
            index += 1
        RESULT_DIR = HERE / f"nn_results_{index:02d}"
    run_single_baseline(PROJECT_DIR, CACHE_DIR, RESULT_DIR, baseline_config=BASELINE_CONFIG)
    print("Résultats enregistrés dans :", RESULT_DIR)

selection = json.loads((RESULT_DIR / "selection.json").read_text())
member_name = selection["members"][0]["name"]
trial = json.loads((RESULT_DIR / "tuning_trials.json").read_text())[0]
holdout = json.loads((RESULT_DIR / "holdout_metrics.json").read_text())
history = json.loads((RESULT_DIR / f"{member_name}_history.json").read_text())
print("Architecture :", selection["members"][0]["config"]["hidden_size"], "neurones cachés")
print("Poids des positifs :", selection["members"][0]["config"]["positive_weight"])
print("Époques retenues :", trial["best_epoch"])
print("Seuil retenu sur le réglage :", selection["threshold"])
print("F1 réglage :", round(trial["tuning"]["f1"], 4))
print("F1 du holdout :", round(holdout["selected_threshold"]["f1"], 4))



La table compare les prédictions avec le seuil choisi sur le réglage, avec le seuil 0,5 et avec la règle « toujours négatif ». Les deux premiers seuils illustrent le compromis entre F1 et accuracy ; le seuil sélectionné est celui fixé pour le CSV.

L'accuracy du modèle doit se lire avec le taux de classe : environ 91 % de ces observations sont négatives. Une accuracy proche de ce taux peut donc cacher un rappel faible. AIcrowd donnera le score du vrai jeu de test après l'envoi.


In [ ]:
print(f"{'Règle':<26} {'F1':>8} {'Accuracy':>10} {'Précision':>10} {'Rappel':>10}")
for label, key in [("Seuil choisi avant holdout", "selected_threshold"),
                   ("Seuil 0.5", "at_05"), ("Toujours négatif", "always_negative")]:
    m = holdout[key]
    print(f"{label:<26} {m['f1']:>8.4f} {m['accuracy']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f}")
m = holdout["selected_threshold"]
print("Matrice de confusion [réel négatif, réel positif] × [prédit négatif, prédit positif] :")
print(np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]]))


In [ ]:
epochs = [row["epoch"] for row in history]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, [row["training_batch_loss"] for row in history], marker="o", label="Loss d'entraînement")
ax2 = ax.twinx()
ax2.plot(epochs, [row["selection_score"] for row in history], marker="s", color="tab:orange", label="F1 réglage")
ax.set(xlabel="Époque", ylabel="Loss (avec pénalité L2)", title=f"Baseline {member_name}")
ax2.set_ylabel("F1 optimisé sur le réglage")
ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(RESULT_DIR / "baseline_nn_courbe.png", dpi=150)
plt.show()



Le fichier contient les **109 379 prédictions**, dans le même ordre que `x_test.csv`, les mêmes IDs, et des labels `-1/+1`. Il a été créé avec `create_csv_submission` de `helpers.py`. Ce fichier est prêt à être téléversé manuellement


In [ ]:
info = json.loads((RESULT_DIR / "submission_info.json").read_text())
print("Fichier :", RESULT_DIR / "submission_nn.csv")
print("Prédictions :", info["rows"])
print("Classe malade prédite :", info["positives"])
print("Soumis à AIcrowd :", info["submitted"])
